# 04. Hybrid 分支的量化（硬件友好 QAT）

对 notebook 03 训练出的 Hybrid 分支做 HW-QAT，对应真实项目的
`train_qat.py --model hybrid_hw`。比 Context 分支（notebook 02）多几处替换：

- `nn.BatchNorm1d` **训练时保留**（在线统计量估计不需要重新发明），导出定点权重时
  **折叠**进前一个卷积层的 weight/bias（教程 06.7 节的折叠公式）。
- `GELU` -> `ReLU6`。
- Jaccard 注意力的 `intersection/union` 除法 -> `fake_quant_reciprocal`，训练时用
  一个伪量化的倒数函数模拟一张真实查找表(101 项,12 位小数精度)的精度。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path("training/semg_snn_90_loop")
sys.path.insert(0, str(PROJECT_ROOT))

from train import EMGDataset
from hw_model import HWHybridSNN, remap_hybrid_state
from hw_fixed_reference import HWFixedHybrid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NB_RUNS = PROJECT_ROOT / "runs_notebook"
WEIGHTS_DIR = NB_RUNS / "weights_hw"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

FP32_HYBRID_CHECKPOINT = NB_RUNS / "hybrid_sja_nb" / "best.pt"
if not FP32_HYBRID_CHECKPOINT.exists():
    FP32_HYBRID_CHECKPOINT = PROJECT_ROOT / "runs" / "hybrid_sja_v1" / "best.pt"
print("Hybrid 热启动源 checkpoint:", FP32_HYBRID_CHECKPOINT)

## 1. 热启动：把 FP32 权重映射进 HW 模型

`remap_hybrid_state` 把 `feature_current.0/1.*` -> `feature_linear/feature_affine.*`，
`conv.front.0/1/3/4.*` -> `conv.conv1/bn1/conv2/bn2.*`（`front` 里的 `GELU` 没有参数,
不需要改名,直接跳过),`fuse.0/1.*` -> `fuse_linear/fuse_affine.*`。

In [ ]:
hw_hybrid_model = HWHybridSNN(features=336).to(device)

fp32_hybrid_state = torch.load(FP32_HYBRID_CHECKPOINT, map_location=device, weights_only=False)["model"]
remapped = remap_hybrid_state(fp32_hybrid_state)
compatible = {k: v for k, v in remapped.items()
              if k in hw_hybrid_model.state_dict() and hw_hybrid_model.state_dict()[k].shape == v.shape}
missing, unexpected = hw_hybrid_model.load_state_dict(compatible, strict=False)
print(f"loaded={len(compatible)}  missing={missing}  unexpected={unexpected}")

## 2. QAT 微调

超参数取自真实项目 `hw_hybrid_qat_v1` 的记录，和 Context 分支一样：
`lr=2e-4, epochs=50, patience=15, weight_power=0.2, label_smoothing=0.03`。

In [ ]:
@torch.no_grad()
def evaluate_hw_hybrid(model, loader):
    model.eval()
    preds, targets = [], []
    for f, raw, y, subject in loader:
        output, _ = model(f.to(device), raw.to(device), subject.to(device))
        preds.extend(output.argmax(1).cpu().tolist())
        targets.extend(y.tolist())
    y_arr, p_arr = np.asarray(targets), np.asarray(preds)
    return {
        "accuracy": accuracy_score(y_arr, p_arr),
        "macro_f1": f1_score(y_arr, p_arr, average="macro"),
        "gesture_accuracy": float(np.mean(p_arr[y_arr != 0] == y_arr[y_arr != 0])),
    }


def train_qat_hybrid(run_name, epochs, lr, patience, weight_power=0.2,
                      label_smoothing=0.03, batch_size=256, seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    sets = {
        split: EMGDataset(
            PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz",
            split == "train", context=1,
        )
        for split in ("train", "val", "test")
    }
    loaders = {
        "train": DataLoader(sets["train"], batch_size, shuffle=True, num_workers=4, pin_memory=True),
        "val": DataLoader(sets["val"], batch_size * 2, num_workers=4, pin_memory=True),
        "test": DataLoader(sets["test"], batch_size * 2, num_workers=4, pin_memory=True),
    }
    optimizer = torch.optim.AdamW(hw_hybrid_model.parameters(), lr=lr, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    counts = np.bincount(sets["train"].y, minlength=13)
    class_weights = torch.tensor((counts.sum() / (13 * counts)) ** weight_power,
                                  dtype=torch.float32, device=device)

    run_dir = NB_RUNS / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_acc, stale = -1.0, 0

    for epoch in range(1, epochs + 1):
        hw_hybrid_model.train()
        losses = []
        for f, raw, y, subject in loaders["train"]:
            f, raw, y, subject = f.to(device), raw.to(device), y.to(device), subject.to(device)
            optimizer.zero_grad(set_to_none=True)
            output, _ = hw_hybrid_model(f, raw, subject)
            loss = nn.functional.cross_entropy(output, y, weight=class_weights,
                                                label_smoothing=label_smoothing)
            loss.backward()
            nn.utils.clip_grad_norm_(hw_hybrid_model.parameters(), 2.0)
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()

        val_metrics = evaluate_hw_hybrid(hw_hybrid_model, loaders["val"])
        print(f"epoch {epoch:02d}  loss={np.mean(losses):.4f}  val_acc={val_metrics['accuracy']:.4f}")
        if val_metrics["accuracy"] > best_acc:
            best_acc, stale = val_metrics["accuracy"], 0
            torch.save({"model": hw_hybrid_model.state_dict(), "epoch": epoch, "validation": val_metrics},
                       run_dir / "best.pt")
        else:
            stale += 1
            if stale >= patience:
                print("early stopping"); break

    ckpt = torch.load(run_dir / "best.pt", map_location=device, weights_only=False)
    hw_hybrid_model.load_state_dict(ckpt["model"])
    test_metrics = evaluate_hw_hybrid(hw_hybrid_model, loaders["test"])
    print(f"\n=== {run_name} 最终结果（第 {ckpt['epoch']} 轮）===")
    print(f"test: accuracy={test_metrics['accuracy']:.4f}  macro_f1={test_metrics['macro_f1']:.4f}")
    return run_dir / "best.pt", test_metrics

qat_hybrid_checkpoint, qat_hybrid_test_metrics = train_qat_hybrid(
    run_name="hw_hybrid_qat_nb", epochs=50, lr=2e-4, patience=15,
)
print("\n参考值(真实项目 hw_hybrid_qat_v1_affinefix): test accuracy≈0.8821")

## 3. FP32 vs QAT 精度对照

In [ ]:
try:
    fp32_row = hybrid_test_metrics  # 来自 notebook 03,如果在同一个 kernel 里连续跑过
except NameError:
    fp32_row = {"accuracy": 0.8876, "macro_f1": 0.7615, "gesture_accuracy": 0.7242}
    print("(未在本 session 里运行过 notebook 03,用 RESULTS.md 记录的参考值代替)\n")

print(f"{'':12s}{'accuracy':>10s}{'macro_f1':>10s}{'gesture_acc':>12s}")
print(f"{'FP32':12s}{fp32_row['accuracy']:10.4f}{fp32_row['macro_f1']:10.4f}{fp32_row['gesture_accuracy']:12.4f}")
print(f"{'HW-QAT':12s}{qat_hybrid_test_metrics['accuracy']:10.4f}"
      f"{qat_hybrid_test_metrics['macro_f1']:10.4f}{qat_hybrid_test_metrics['gesture_accuracy']:12.4f}")

## 4. 导出定点权重 + numpy 参考实现交叉验证

对 Hybrid 分支重复 notebook 02 第 6 节做过的事：导出 `int8 codes + scale`，
额外要处理 BatchNorm 折叠和倒数查找表的构建（教程 06.7/06.8 节）。

In [ ]:
def export_hybrid_fixed(state: dict, weight_bits: int, out_path: Path) -> None:
    """精简版 export_hw_fixed.py::export_hybrid()。"""
    arrays = {}

    def add_linear(prefix, weight_key, bias_key, bits):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 2 ** (bits - 1) - 1
        amax = np.maximum(np.abs(weight).max(axis=tuple(range(1, weight.ndim)), keepdims=True), 1e-8)
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = scale.reshape(-1).astype(np.float32)
        arrays[f"{prefix}_bias"] = bias

    def add_affine(prefix, weight_key, bias_key):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 127
        amax = float(np.maximum(np.abs(weight).max(), 1e-8))
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = np.float32(scale)
        arrays[f"{prefix}_bias"] = bias

    def add_bn(prefix):
        weight = state[f"conv.{prefix}.weight"].detach().cpu().numpy().astype(np.float32)
        bias = state[f"conv.{prefix}.bias"].detach().cpu().numpy().astype(np.float32)
        mean = state[f"conv.{prefix}.running_mean"].detach().cpu().numpy().astype(np.float32)
        var = state[f"conv.{prefix}.running_var"].detach().cpu().numpy().astype(np.float32)
        bn_scale = weight / np.sqrt(var + 1e-5)
        arrays[f"{prefix}_scale"] = bn_scale
        arrays[f"{prefix}_shift"] = bias - mean * bn_scale

    add_linear("feature_linear", "feature_linear.weight", "feature_linear.bias", weight_bits)
    add_affine("feature_affine", "feature_affine.weight", "feature_affine.bias")
    add_linear("conv1", "conv.conv1.weight", "conv.conv1.bias", weight_bits); add_bn("bn1")
    add_linear("conv2", "conv.conv2.weight", "conv.conv2.bias", weight_bits); add_bn("bn2")
    add_linear("q", "conv.q.weight", "conv.q.bias", weight_bits)
    add_linear("k", "conv.k.weight", "conv.k.bias", weight_bits)
    add_linear("v", "conv.v.weight", "conv.v.bias", weight_bits)
    add_linear("fuse_linear", "fuse_linear.weight", "fuse_linear.bias", weight_bits)
    add_affine("fuse_affine", "fuse_affine.weight", "fuse_affine.bias")
    add_linear("out", "out.weight", "out.bias", weight_bits)

    def quantize_decay(beta, frac_bits=8):
        scale = 2.0 ** (-frac_bits)
        return float(np.clip(np.round(beta / scale), 1, 2 ** frac_bits) * scale)

    arrays["beta_f"] = np.float32(quantize_decay(float(torch.sigmoid(state["beta_f"]))))
    arrays["beta_o"] = np.float32(quantize_decay(float(torch.sigmoid(state["beta_o"]))))
    arrays["conv_beta"] = np.float32(quantize_decay(float(torch.sigmoid(state["conv.beta"]))))

    # Jaccard 倒数查找表: 101 项, 12 位小数精度 (教程 06.8 节)
    values = np.arange(0, 101)
    denom = np.maximum(values, 1)
    recip_scale, recip_limit = 2.0 ** (-12), 2 ** 12
    arrays["reciprocal_table"] = (np.clip(np.round((1.0/denom)/recip_scale), 0, recip_limit) * recip_scale).astype(np.float32)
    arrays["act_frac_bits"], arrays["act_int_bits"] = np.int32(8), np.int32(8)
    arrays["steps"], arrays["time_steps"] = np.int32(12), np.int32(100)

    np.savez(out_path, **arrays)
    total_bytes = sum(a.nbytes for a in arrays.values() if hasattr(a, "nbytes"))
    print(f"wrote {out_path} ({total_bytes/1024:.1f} KiB)")

export_hybrid_fixed(hw_hybrid_model.state_dict(), weight_bits=4, out_path=WEIGHTS_DIR / "hw_hybrid_fixed_nb.npz")

# 交叉验证 (同 notebook 02 第 6 节的方法,这次针对 Hybrid)
numpy_hybrid = HWFixedHybrid(WEIGHTS_DIR / "hw_hybrid_fixed_nb.npz")
val_dataset = EMGDataset(PROJECT_ROOT / "data" / "val.npz", PROJECT_ROOT / "data" / "normalization.npz",
                          False, context=1)
sample_idx = np.random.RandomState(0).choice(len(val_dataset), size=128, replace=False)
feats = torch.stack([val_dataset[i][0] for i in sample_idx]).to(device)
raws = torch.stack([val_dataset[i][1] for i in sample_idx]).to(device)

hw_hybrid_model.eval()
with torch.no_grad():
    torch_logits, _ = hw_hybrid_model(feats, raws, None)
numpy_logits, numpy_argmax = numpy_hybrid.infer(feats.cpu().numpy(), raws.cpu().numpy())
match_rate = (torch_logits.cpu().numpy().argmax(1) == numpy_argmax).mean()
print(f"\nHybrid argmax 一致率: {match_rate:.4f}  (预期 1.0)")

## 下一步

Context、Hybrid 两个分支都完成了训练+量化。打开 [05_delay_training.ipynb](05_delay_training.ipynb)
处理第三个、也是结构最简单的分支——Delay-SNN。它的量化方式会和这两个分支很不一样
（不需要重新训练），值得对比着看。